# 🚌 School Transport Route Optimization & Geospatial Query System

## Executive Summary
This project delivers an automated spatial analytics pipeline and interactive tool designed to solve K-12 school transport logistics challenges. By parsing multi-route Excel datasets and implementing the **Haversine trigonometric distance formula**, this system enables transport managers to instantly query student/parent coordinates against existing bus stops to assign the nearest pickup point, calculate direct distances, and map optimal routing vectors.

--- 

## Key Features & Operational Highlights
- **Multi-Sheet Automated Ingestion**: Reads and standardizes route datasets across Senior and Junior operational wings.
- **Geodesic Distance Engine**: Uses pure Python mathematical routines (Haversine formula) to calculate distances in kilometers without third-party API costs or latency.
- **Privacy & Anonymization Pipeline**: Strips confidential Student PII (Names, Phone Numbers, Exact Flat Numbers) down to generalized public sector hubs and landmark markers.
- **Interactive Mapping**: Generates Folium map layers and dashed vector lines indicating direct proximity.

In [ ]:
import math
import pandas as pd
import folium

print("Libraries successfully loaded.")

## 1. Data Ingestion & Sanitization Engine
The function below loads all sheet tabs from the Excel route file, standardizes coordinates, assigns operational categories (`Senior` vs `Junior`), and extracts sector-level stop locations while ensuring personal student details are not exposed.

In [ ]:
def load_and_sanitize_routes(excel_path):
    excel_file = pd.ExcelFile(excel_path)
    combined_stops = []
    
    for sheet in excel_file.sheet_names:
        df = pd.read_excel(excel_path, sheet_name=sheet)
        
        if 'Latitude' not in df.columns or 'Longitude' not in df.columns:
            continue
            
        df = df.dropna(subset=['Latitude', 'Longitude']).copy()
        df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
        df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
        df = df.dropna(subset=['Latitude', 'Longitude'])
        
        sheet_upper = sheet.strip().upper()
        if '(S)' in sheet_upper or 'S ' in sheet_upper:
            cat = 'Senior'
        elif '(J)' in sheet_upper or 'J ' in sheet_upper or ')J' in sheet_upper:
            cat = 'Junior'
        else:
            cat = 'Senior' if 'S' in sheet_upper else 'Junior'
            
        df['Route_Name'] = sheet.strip()
        df['Category'] = cat
        
        if 'Address' in df.columns:
            df['Stop_Name'] = df['Address'].astype(str)
        else:
            df['Stop_Name'] = 'Sector Pickup Hub'
            
        combined_stops.append(df[['Route_Name', 'Category', 'Stop_Name', 'Latitude', 'Longitude']])
        
    return pd.concat(combined_stops, ignore_index=True)

print("Sanitization pipeline defined successfully.")

## 2. Geodesic Proximity Engine (Haversine Formula)
To compute exact spatial distance between two spherical points $(lat_1, lon_1)$ and $(lat_2, lon_2)$ without calling paid external geocoding APIs, we implement the Haversine equation:

$$a = \sin^2\left(\frac{\Delta lat}{2}\right) + \cos(lat_1) \cdot \cos(lat_2) \cdot \sin^2\left(\frac{\Delta lon}{2}\right)$$
$$c = 2 \cdot \arcsin(\sqrt{a})$$
$$d = R \cdot c$$

where $R = 6371.0 \text{ km}$ (Earth's mean radius).

In [ ]:
def calculate_haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1_rad, lon1_rad = math.radians(lat1), math.radians(lon1)
    lat2_rad, lon2_rad = math.radians(lat2), math.radians(lon2)
    
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = (math.sin(dlat / 2) ** 2 + 
         math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2) ** 2)
    return R * 2 * math.asin(math.sqrt(a))

def find_nearest_pickup_stop(parent_lat, parent_lon, df_stops, category_filter='All'):
    if category_filter in ['Senior', 'Junior']:
        filtered_df = df_stops[df_stops['Category'] == category_filter]
    else:
        filtered_df = df_stops
        
    min_distance = float('inf')
    nearest_stop = None
    
    for _, row in filtered_df.iterrows():
        dist = calculate_haversine_distance(parent_lat, parent_lon, row['Latitude'], row['Longitude'])
        if dist < min_distance:
            min_distance = dist
            nearest_stop = row
            
    return nearest_stop, min_distance

print("Spatial query engine ready.")

## 3. Interactive Route Mapping & Proximity Rendering
The function below generates a interactive Folium map highlighting the parent's residence, existing bus stop hubs, and a dashed line connecting the parent to the nearest route stop.

In [ ]:
def generate_inquiry_map(parent_lat, parent_lon, nearest_stop, min_distance, df_stops):
    m = folium.Map(location=[parent_lat, parent_lon], zoom_start=13, tiles='OpenStreetMap')
    
    folium.Marker(
        [parent_lat, parent_lon],
        popup="Parent Inquiry Location",
        icon=folium.Icon(color='red', icon='home', prefix='fa')
    ).add_to(m)
    
    if nearest_stop is not None:
        folium.Marker(
            [nearest_stop['Latitude'], nearest_stop['Longitude']],
            popup=f"<b>{nearest_stop['Route_Name']}</b><br>{nearest_stop['Stop_Name']}<br>Distance: {min_distance:.2f} km",
            icon=folium.Icon(color='green', icon='bus', prefix='fa')
        ).add_to(m)
        
        folium.PolyLine(
            [[parent_lat, parent_lon], [nearest_stop['Latitude'], nearest_stop['Longitude']]],
            color='green',
            weight=3,
            dash_array='5,5',
            tooltip=f"{min_distance:.2f} km to {nearest_stop['Route_Name']}"
        ).add_to(m)
        
    return m

print("Map generation engine ready.")